# 3. Search the maps and inspect a candidate catalog

**Scenario:** you have PIRATE's saved S/N and argmax maps. You want a list of candidate signals with physical DM, arrival time, S/N, and filter width.

Use the maps from [notebook 2](02_dedispersion.ipynb). Start the Bash terminal and Jupyter kernel in this tutorial repository. Set the blank RUN_DIR and run_directory fields to the **same full run-folder path used in notebooks 1 and 2**.

Read the config, run the search in your activated GPU terminal, and wait for completion. Python below only reads and visualizes the results. The instructor has reported a successful local walkthrough; this standalone edition and scientific recovery checks still need validation.

## 1. Read the search configuration

Open grouper.yml inside your run folder, copied from [configs/grouper.yml](configs/grouper.yml):

~~~yaml
peakfinding:
  snr_threshold: 10.0
  dm_reach: 8
  waist_bins: 1
grouping:
  halo_size: 2
  dm_tolerance: 1.5
  time_tolerance: 1.5
execution:
  beam_batch_size: 1
  timeout_ms: 0
  timeout_policy: discard
~~~

For this exercise, focus on **snr_threshold**. Leave the native neighborhood, grouping, and execution settings unchanged. The search over trial DMs and filter profiles already happened during dedispersion in notebook 2; this stage completes candidate extraction.

PIRATE follows this sequence:

1. Compare map pixels with their neighbors and apply the peak-selection rules and S/N threshold.
2. Decode each retained pixel's argmax token using the saved producer metadata.
3. Group related detections and retain the loudest member as each group's representative.

A **detection** is a retained map peak. Several detections can describe one physical pulse. The saved catalog calls the grouped candidates **events**, and the detections assigned to them **members**. A candidate is a signal to inspect, not proof of an astrophysical source.

## 2. Run the search in the terminal

Fill in RUN_DIR with the same full run-folder path used in notebooks 1 and 2, then run this command in your activated GPU terminal. The ${RUN_DIR:?...} check prevents running with a blank path.

~~~bash
RUN_DIR=""  # REQUIRED: full path to the existing run folder from notebook 1.
python -m pirate_frb run offline_grouper \
    "${RUN_DIR:?Fill in RUN_DIR first}/frames" "$RUN_DIR/grouper.yml" \
    --output "$RUN_DIR/events.asdf"
~~~

**--output is a file path, not a directory.** This command creates events.asdf inside your run folder; do not create a directory named events.asdf.

PIRATE performs peak finding, decoding, and grouping on the GPU, and writes the catalog. It processes the complete acquisition so groups near chunk boundaries can finish. Wait for successful completion before continuing.

The catalog records processing settings and the source coordinates of each event. Rerunning with the same --output path replaces that catalog; use a new filename when comparing thresholds. The input frames and maps can be reused unchanged.


## 3. Read the catalog in Python

PIRATE's catalog validator checks the saved format before we display its columns.

The default injection was DM 20 pc cm^-3, arrival 5 s at 300 MHz, Gaussian sigma 3 ms, and S/N 200. Open generation.log inside your run folder to confirm the values for **your** run, especially after changing the injection command.

In [ ]:
from pathlib import Path
import asdf
import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import Markdown, display
from pirate_frb.TriggerCatalog import validate_trigger_catalog_tree
from pirate_frb.BowtiePeakfinding import EdgeFlag


In [ ]:
run_directory = ""  # REQUIRED: full path to the run folder you chose in the terminal.
if not run_directory.strip():
    raise ValueError("Fill in run_directory above before continuing.")
run_dir = Path(run_directory).expanduser()
catalog_path = run_dir / 'events.asdf'  # A file, not a directory.


In [ ]:
with asdf.open(catalog_path) as catalog:
    validate_trigger_catalog_tree(catalog.tree)
    events = {key: np.asarray(value).copy() for key, value in catalog.tree['events'].items()}
    member_count = len(catalog.tree['members']['candidate_id'])
    producer_config = yaml.safe_load(catalog.tree['metadata']['producer']['config_yaml'])
dt = producer_config['time_sample_ms'] * 1e-3
arrival_s = events['toa_sample_abs'] * dt
print(f'{len(events["event_id"])} grouped candidates; {member_count} member detections')

### What do the columns mean?

The decoded toa_sample_abs is a pulse-center time in native samples. This acquisition starts at sample zero, so multiplying by the recorded sampling interval gives seconds since observation start, referenced to **300 MHz**.

The recovered width_ms is the selected **nominal filter width**. PIRATE searches different profile shapes; this field is neither necessarily the full filter duration nor the injected Gaussian sigma.

Recovered S/N need not equal the injected value: noise, quantization, and the finite set of DM/time/filter trials affect the result. Fine coordinates decoded from the argmax token provide more detail than the coarse plot pixels in notebook 2.

In [ ]:
table = ['| Event | Beam | Arrival at 300 MHz (s) | DM (pc cm^-3) | S/N | Nominal filter width (ms) | Members |',
         '| --- | --- | --- | --- | --- | --- | --- |']
for i in range(len(events['event_id'])):
    table.append(
        f"| {events['event_id'][i]} | {events['beam_id'][i]} | {arrival_s[i]:.6f} | "
        f"{events['dm'][i]:.3f} | {events['snr'][i]:.2f} | "
        f"{events['width_ms'][i]:.3f} | {events['member_count'][i]} |")
display(Markdown('\n'.join(table)))

**What to look for:** a strong candidate close to the injected DM and arrival time. Use the table and generation.log to compare its S/N and nominal filter width with the injected S/N and Gaussian sigma. Do not require exact equality or exactly one catalog row.

## 4. Inspect one candidate

Choose the loudest candidate as a starting point. Being loudest alone does not establish a match: check its DM, time, beam, and quality flags.

The next cell prints PIRATE's flag names. If a flag is present or grouping timed out, review the [native definitions](https://github.com/MattTrudu/pirate/blob/c7269d69ae7327bac71793b780f78cc3c2d15cc8/pirate_frb/BowtiePeakfinding.py) with your instructor before interpreting recovery. For example, STARTUP_INCOMPLETE warns that missing earlier data may bias S/N and width.

In [ ]:
if len(events['event_id']) == 0:
    raise ValueError('No candidates: inspect the threshold and the saved S/N map in notebook 2.')
event_row = int(np.argmax(events['snr']))
event = {key: values[event_row].item() for key, values in events.items()}
print(f"Selected event {event['event_id']}, beam {event['beam_id']}")
print(f"DM: {event['dm']:.3f} pc cm^-3; arrival at 300 MHz: {arrival_s[event_row]:.6f} s")
print(f"Recovered S/N: {event['snr']:.2f}; nominal filter width: {event['width_ms']:.3f} ms")
print(f"Quality: {EdgeFlag(event['edge_flags']).name or 'none'}; grouping_timed_out={event['grouping_timed_out']}")

## 5. Find the candidate's source pixel

The catalog preserves the beam, tree, source chunk, DM-row index (idm), time-column index (itime), and argmax token of its representative.

Use those saved indices to locate the source pixel directly. **Do not convert the decoded arrival time or DM back to coarse indices:** they include finer information and filter timing conventions.

The token printed below is only an identifier for the winning fine trial. We visualize its associated S/N, not the integer token as a brightness.

In [ ]:
map_name = f"frame_b{event['beam_id']}_t{event['source_chunk_index']}_snrmap.asdf"
with asdf.open(run_dir / 'frames' / map_name) as saved:
    snr = np.asarray(saved.tree['trees'][event['tree']]['snr']).copy()
idm, itime = event['idm'], event['itime']
print(f"Source: {map_name}, tree {event['tree']}, pixel ({idm}, {itime})")
print(f"Saved winning token: 0x{event['argmax_token']:08x}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
image = ax.imshow(snr, origin='lower', aspect='auto', interpolation='nearest', cmap='magma')
ax.scatter(itime, idm, marker='+', color='cyan', s=150, label='Representative pixel')
ax.set(xlim=(max(-0.5, itime - 20.5), min(snr.shape[1] - 0.5, itime + 20.5)),
       ylim=(max(-0.5, idm - 24.5), min(snr.shape[0] - 0.5, idm + 24.5)),
       xlabel='Time column (index)', ylabel='DM row (index)',
       title=f"Source-map neighborhood of event {event['event_id']}")
ax.legend()
fig.colorbar(image, ax=ax, label='Filtered S/N')
plt.show()

**What to look for:** the marker identifies the precise coarse S/N pixel from which PIRATE decoded this event. These axes are array indices; the candidate table contains the physical measurements. The broader bright neighborhood can contain responses from the same injected pulse.

### Try one change

First record the baseline member/candidate counts and the selected event's DM, time, and S/N, or save a notebook copy with its outputs.

In the same terminal, keep RUN_DIR set to the observation you are using, including if you chose the fainter pulse. Copy its grouper config:

~~~bash
cp "${RUN_DIR:?Set RUN_DIR to your existing run folder}/grouper.yml" "$RUN_DIR/grouper_threshold15.yml"
~~~

Use the Jupyter text editor to change **only peakfinding.snr_threshold from 10 to 15** in the copied file. Then run:

~~~bash
python -m pirate_frb run offline_grouper \
    "${RUN_DIR:?Set RUN_DIR to your existing run folder}/frames" "$RUN_DIR/grouper_threshold15.yml" \
    --output "$RUN_DIR/events_threshold15.asdf"
~~~

Wait for completion, change catalog_path in the Python path cell to run_dir / 'events_threshold15.asdf', then rerun the reading/plotting cells.

Predict which weaker detections might disappear. Compare member counts and the bright pulse's recovery. Counts may stay unchanged in this clean example, and regrouping means grouped-event counts need not change monotonically. **No new injection or dedispersion is needed.**

You have followed one saved acquisition through three terminal commands: generation, dedispersion, and candidate search. For a more detailed diagnostic workflow, see PIRATE's [classifier-window notebook](https://github.com/MattTrudu/pirate/blob/c7269d69ae7327bac71793b780f78cc3c2d15cc8/AIclassifier/README.md).
